# LAB 6 : L’état et le contexte d’un agent

Ce notebook présente l’utilisation du contexte et de l’état dans un agent LangChain.

Le TP est organisé comme suit :

1. Définir une classe de contexte structurée `ColourContext`
2. Créer un agent sans outils utilisant un contexte
3. Créer un agent avec des outils qui lisent le contexte
4. Modifier le contexte au moment de l’appel
5. Définir un état personnalisé avec `AgentState`
6. Créer un agent qui modifie l’état
7. Créer un agent qui récupère l’état

#### PARTIE 1 : Définir une classe de contexte structurée appelée ColourContext

In [1]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

#### PARTIE 2 : Agent sans contexte

In [2]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

model = ChatOllama(
    model="llama3.2:3b",  # ou mistral, gemma, etc.
)

agent = create_agent(
    model=model,
    context_schema=ColourContext
)

c:\Users\Najlaa\Documents\TPBDCC\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

print(response['messages'][-1].content)

I don't have any information about your personal preferences or characteristics. I'm a text-based AI assistant, and our conversation just started. Would you like to share your favourite colour with me?


#### PARTIE 3 : Agent avec contexte

In [4]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

In [5]:
@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [6]:
agent = create_agent(
    model=model,
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [7]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

print(response['messages'][-1].content)

c:\Users\Najlaa\Documents\TPBDCC\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  function=lambda v, h: h(v), schema=original_schema
c:\Users\Najlaa\Documents\TPBDCC\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


The output from the tool call suggests that your favourite colour is blue, so I'll respond with a simple confirmation:

Your favourite colour is indeed blue!


#### PARTIE 4 : Changement de contexte

In [8]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")
)

print(response['messages'][-1].content)

c:\Users\Najlaa\Documents\TPBDCC\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  function=lambda v, h: h(v), schema=original_schema
c:\Users\Najlaa\Documents\TPBDCC\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


The user's favourite colour is green.


#### PARTIE 5 : Définir un état personnalisé d’agent en héritant de AgentState

In [11]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

#### PARTIE 6 : Agent qui modifie un état

In [12]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour,
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]
    })

In [13]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [14]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

print(response['messages'][-1].content)

It looks like the tool call was successful, and I've successfully updated your favourite colour to green. If you have any other questions or need assistance with anything else, feel free to ask!


#### PARTIE 7 : Agent qui récupère un état

In [17]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

In [18]:
response = agent.invoke(
    {"messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

print(response['messages'][-1].content)

I've successfully updated your favourite colour to green again. It seems that this information has been retained from our previous conversation. If you'd like to change it or add more information, just let me know!


In [19]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

print(response['messages'][-1].content)

It looks like I made a mistake again! It seems that the tool call didn't actually update anything, and your favourite colour is still green. To confirm, your current favourite colour is indeed green. Is there anything else I can help you with?
